In [0]:
%sql

---------------------create owener list-----------------
select distinct cast(CMS.Document_Id as string) as Doc_ID, (case when S1.document_id is not null then true else false end) as scheduled, CL.cluster, CMS.created as Owner, UP.DisplayName, UP.Title, UP.BU, UP.BU_Old, UP.Country, CMS.updated as last_update_by
from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata as CMS
left join dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_cluster_details as CL
on upper(trim(CL.Document_ID))=upper(trim(CMS.Document_Id ))
left join dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile as UP
on UP.UserName = CMS.created
left join dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_schedule_extracted as S1
on upper(trim(CMS.Document_Id))=upper(trim(S1.document_id))
where CMS.Document_Id is not null 
  and cms.Document_name not in ('Document Not Found on Server')
  and UP.BU =''

select distinct cast(cluster as string), scheduled, count(distinct Doc_ID) from _sqldf
group by cluster, scheduled
union all 
select "Total" as cluster,  count(distinct Doc_ID) from _sqldf
group by scheduled
order by cluster asc, scheduled asc


select distinct cast(CL.Document_ID as string) as Doc_ID, CL.cluster, CL.created as Owner, UP.DisplayName, UP.Title, UP.BU, UP.BU_Old, UP.Country, CMS.updated as last_update_by
from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata as CMS
left join dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_cluster_details as CL
on upper(trim(CL.Document_ID))=upper(trim(CMS.Document_Id ))
left join dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile as UP
on UP.UserName = CMS.created
where CMS.created is null

select distinct cms.Document_Id, cms.Document_name,cms.WEBI_Found_on_Server,cms.Full_path,cms.ingestion_ts from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata  as cms
left join dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.sapbo_full_doc_id as sf
on upper(Trim(cms.Document_Id))=upper(trim(sf.Report_ID))
where sf.Report_ID is null
and cms.Document_name not in ('Document Not Found on Server')

select 
-- select * from dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_schedule_extracted

In [0]:
%sql
select cluster, BU, scheduled, count(distinct Doc_ID) as document_cnt, count(distinct UserName) as user_cnt  
from dataplatform01_central_dev_catalog_01.custom_sap_bo.cluster_analysis
GROUP BY cluster, BU, scheduled
having count(distinct UserName) > 0
;

select count(distinct Doc_ID) from dataplatform01_central_dev_catalog_01.custom_sap_bo.cluster_analysis


In [0]:
import matplotlib.pyplot as plt
import pandas as pd

pdf = spark.sql("""
select cluster, BU, scheduled, count(distinct Doc_ID) as document_cnt, count(distinct UserName) as user_cnt  
from dataplatform01_central_dev_catalog_01.custom_sap_bo.cluster_analysis
GROUP BY cluster, BU, scheduled
having count(distinct UserName) > 0
""").toPandas()

# Pivot: cluster on x-axis, BU as legend
pivot_df = pdf.groupby(['cluster', 'BU'])['document_cnt'].sum().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(14, 7))
pivot_df.plot(kind='bar', stacked=True, ax=ax, colormap='tab20')
ax.set_xlabel('Cluster')
ax.set_ylabel('Document Count')
ax.set_title('Document Count by Cluster and BU')
ax.legend(title='BU', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()

# Pie chart: BU distribution for Cluster 6
cluster6 = pdf[pdf['cluster'] == 6.0].groupby('BU')['document_cnt'].sum()
cluster6 = cluster6[cluster6 > 0].sort_values(ascending=False)

fig2, ax2 = plt.subplots(figsize=(10, 8))
ax2.pie(cluster6, labels=cluster6.index, autopct='%1.1f%%', startangle=140, textprops={'fontsize': 8})
ax2.set_title('Cluster 6 - BU Distribution (Document Count)')
plt.tight_layout()
plt.show()